# Project 01: Hybrid Search RAG with LangChain & Pinecone

This notebook provides a step-by-step walkthrough of **Hybrid Search Retrieval-Augmented Generation (RAG)**.

### Why Hybrid Search?
- **Dense Retrieval (Semantic Search)**: Excellent at catching contextual and semantic similarity (e.g., mapping "dog" to "canine"). However, it struggles with specific keywords, product codes, or rare terms.
- **Sparse Retrieval (Keyword Search)**: Highly accurate for exact matching (e.g., looking up specific part numbers or acronyms like "BM25"). But it completely misses synonyms.

**Hybrid Search** combines both dense and sparse representations to produce a unified relevance score. Let's see how they are calculated and used together.

## 1. Environment and Library Imports

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
import nltk

# Add parent directory to path to enable local src imports
sys.path.append(str(Path.cwd() / "src"))

# Load .env file
load_dotenv()

# Ensure NLTK resources are available
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

print("Libraries loaded successfully.")

## 2. Exploring Sparse Embeddings (BM25)

Sparse vectors represent text based on term frequency and document frequency. The dimensions correspond to words in a vocabulary, and the values represent weights (usually BM25 score).

Let's fit a BM25 encoder on sample texts and examine its output.

In [ ]:
from pinecone_text.sparse import BM25Encoder

# Sample document corpus
corpus = [
    "Retrieval-Augmented Generation (RAG) is a pattern that enhances LLMs with external knowledge bases.",
    "Dense retrieval uses deep learning models like OpenAI's text-embedding-3-small to embed text.",
    "Sparse retrieval uses traditional keyword-matching algorithms like BM25."
]

# Initialize and train the encoder
bm25_encoder = BM25Encoder()
bm25_encoder.fit(corpus)

print(f"Vocab size: {len(bm25_encoder.dictionary)}")
print("Sample words in vocabulary:", list(bm25_encoder.dictionary.keys())[:10])

In [ ]:
# Encode a query into a sparse representation
query = "What is BM25 and sparse retrieval?"
sparse_vector = bm25_encoder.encode_queries(query)

print("Encoded Sparse Vector:")
print(sparse_vector)

Notice that the sparse vector consists of list of `indices` and list of `values`. 
- `indices` correspond to specific word indices in the trained vocabulary. 
- `values` represent the calculated BM25 weights for those words.

## 3. Exploring Dense Embeddings (OpenAI)

Dense vectors represent text in a continuous, high-dimensional space where distance translates to semantic similarity.

*(Requires `OPENAI_API_KEY` set in your `.env`)*

In [ ]:
from langchain_openai import OpenAIEmbeddings

try:
    # OpenAI embeddings initialization
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    dense_vector = embeddings.embed_query("What is BM25 and sparse retrieval?")
    
    print(f"Dense Vector Dimensions: {len(dense_vector)}")
    print("First 10 values:", dense_vector[:10])
except Exception as e:
    print("⚠️ API Key Error:", e)
    print("Please set a valid OPENAI_API_KEY in your .env file to generate dense embeddings.")

## 4. Connecting to Pinecone & Building the Hybrid Index

Let's see how both vectors are passed to Pinecone to construct a Hybrid RAG pipeline.

*(Requires both `OPENAI_API_KEY` and `PINECONE_API_KEY` configured)*

In [ ]:
from pinecone import Pinecone, ServerlessSpec

pinecone_api_key = os.getenv("PINECONE_API_KEY")
index_name = os.getenv("PINECONE_INDEX_NAME", "hybrid-search-rag")

if not pinecone_api_key or "your_" in pinecone_api_key:
    print("⚠️ Configure your PINECONE_API_KEY and OPENAI_API_KEY in the .env file to run this section.")
else:
    pc = Pinecone(api_key=pinecone_api_key)
    existing_indexes = [idx.name for idx in pc.list_indexes()]
    
    if index_name not in existing_indexes:
        print(f"Creating index {index_name}...")
        pc.create_index(
            name=index_name,
            dimension=1536,
            metric="dotproduct",  # dotproduct metric is required for hybrid search!
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )
    else:
        print(f"Index '{index_name}' ready.")

## 5. Integrating with LangChain and Querying

Now we set up LangChain's Hybrid Search Retriever and link it with a Chat model to form a complete RAG system.

In [ ]:
from langchain_community.retrievers import PineconeHybridSearchRetriever
from langchain_openai import ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

if not pinecone_api_key or "your_" in pinecone_api_key:
    print("⚠️ Please configure API keys first.")
else:
    index = pc.Index(index_name)
    
    # Initialize retriever
    retriever = PineconeHybridSearchRetriever(
        embeddings=embeddings,
        sparse_encoder=bm25_encoder,
        index=index
    )
    
    # Run a test retrieval query
    query_str = "RAG external knowledge"
    docs = retriever.invoke(query_str)
    
    print(f"--- Retrieved Documents for '{query_str}' ---")
    for i, doc in enumerate(docs):
        print(f"\nDocument [{i+1}]:")
        print(doc.page_content)
        print("Metadata:", doc.metadata)

## 6. Complete Generation Chain (RAG)

Let's ask the LLM a question and fetch the synthesized response utilizing the retrieved context.

In [ ]:
if not pinecone_api_key or "your_" in pinecone_api_key:
    print("⚠️ Please configure API keys first.")
else:
    # RAG prompt
    prompt_template = ChatPromptTemplate.from_template(
        """Answer the user's question using the context below. If you don't know, state that you don't know.
        
        Context:
        {context}
        
        Question: {input}
        
        Answer:"""
    )
    
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    # Build chains using LangChain LCEL wrappers
    combine_docs_chain = create_stuff_documents_chain(llm, prompt_template)
    rag_chain = create_retrieval_chain(retriever, combine_docs_chain)
    
    # Run QA
    question = "What advantages does hybrid search offer over standard search?"
    response = rag_chain.invoke({"input": question})
    
    print(f"\nQuestion: {question}")
    print(f"\nAnswer:\n{response['answer']}")